DATA Ingestion 


In [1]:
from langchain_core.documents import Document

In [2]:
doc = Document(
    page_content = "hi this is leo abinab and i am creating an rag file",
    metadata = {
        "source": "example.txt",
        "authore": "leo abinab",
        "date_created" : "12-03-2026"
    }
)
doc

Document(metadata={'source': 'example.txt', 'authore': 'leo abinab', 'date_created': '12-03-2026'}, page_content='hi this is leo abinab and i am creating an rag file')

In [10]:
import os
os.makedirs("../data/text_files", exist_ok = True)

In [4]:
sample_text = {
    "../data/text_files/python_intro.txt": """Python Programming Introduction
Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.
Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support"""
}

for filepath,content in sample_text.items() : 
    with open(filepath, 'w', encoding="utf-8") as f:
        f.write(content)

print("sample file created")

sample file created


In [5]:
###loading the text

from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python_intro.txt")
document = loader.load()
print(document)

C:\Users\MADHU\AppData\Local\Temp\ipykernel_26560\1339380722.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support')]


In [14]:
### loading all the pdf

from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

dir_loaders = DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",
    loader_cls= PyMuPDFLoader,
    show_progress=False
)

pdf_documents = dir_loaders.load()
pdf_documents

# 1. Initialize the Text Splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,     # Size of each text chunk (in characters)
    chunk_overlap=50    # Overlap between consecutive chunks to keep context
)

# 2. Perform chunking on your loaded PDF documents
chunked_documents = text_splitter.split_documents(pdf_documents)

print(f"Original Pages: {len(pdf_documents)}")
print(f"Total Text Chunks Created: {len(chunked_documents)}")


C:\Users\MADHU\AppData\Local\Temp\ipykernel_19276\1952358234.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader


Original Pages: 59
Total Text Chunks Created: 362


Chuncking part


In [15]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [16]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()


    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        Args:
            texts: List of text strings to embed
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

embedding_manager = EmbeddingManager()
embedding_manager
    

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8946.97it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\MADHU\AppData\Local\Temp\ipykernel_19276\4677350.py:21: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


Vector Store

In [17]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()


    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
        

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            # Document content
            documents_text.append(doc.page_content)
            # Embedding
            embeddings_list.append(embedding.tolist())
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore



Vector store initialized. Collection: pdf_documents
Existing documents in collection: 367


In [18]:
chunked_documents

text = [ doc.page_content for doc in chunked_documents]
text

embeddings = embedding_manager.generate_embeddings(text)

vectorstore.add_documents(chunked_documents, embeddings)

Generating embeddings for 362 texts...


Batches: 100%|██████████| 12/12 [00:14<00:00,  1.24s/it]


Generated embeddings with shape: (362, 384)
Adding 362 documents to vector store...
Successfully added 362 documents to vector store
Total documents in collection: 729


In [19]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager


    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            retrieved_docs = []
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
            print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            return retrieved_docs
        except Exception as e:
            print(f"Error querying vector store: {e}")
            raise


In [20]:
rag_retriver = RAGRetriever(vectorstore, embedding_manager)
rag_retriver.retrieve("what is Agentic RAG — retrieval as a reasoning loop")

Retrieving documents for query: 'what is Agentic RAG — retrieval as a reasoning loop'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 58.60it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_99bae734_326',
  'content': 'retrieval saves significant compute and also reduces retrieval noise — fewer\nirrelevant passages contaminating responses to questions the model already had the\nanswer to. (arXiv:2403.14403)\n8. Agentic RAG — retrieval as a reasoning loop\nAll previous RAG types follow a predetermined script: retrieve once (or a fixed\nnumber of times), then generate. Agentic RAG abandons the script. An LLM agent\ncontrols the retrieval process entirely: decide what it needs to know, choose which',
  'metadata': {'source': '..\\data\\pdf\\medium-article-2.pdf',
   'total_pages': 19,
   'title': 'Retrieval-Augmented Generation (RAG): The Complete Guide | by Kunal Khanvilkar | Apr, 2026 | Medium',
   'file_path': '..\\data\\pdf\\medium-article-2.pdf',
   'creationDate': "D:20260412193411+00'00'",
   'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
   'format': 'PDF 1.4',
   'producer': 